# <font color="#F48E16">XAI aplicada al Deep Learning: Ejercicio práctico: XAI con RNNs y análisis de sentimiento con IMDB</font>

Material generado por <a href="https://www.linkedin.com/in/christian-oliva-moya-ingeniero/">Christian Oliva</a>. Cualquier duda, sugerencia o errata, no duden en contactar.

**Versión 1.0** - 12 de mayo de 2026

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tqdm import tqdm
from time import time

from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PATH = "/content/drive/MyDrive/CURSO XAI/BLOQUE 2 - Explicabilidad/"

## <font color="#F48E16">Descarga y preprocesamiento de datos</font>

In [ ]:
input_size = 10000
max_len = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)
x_train = pad_sequences(x_train, maxlen=max_len, padding="post", truncating="post")
x_test = pad_sequences(x_test, maxlen=max_len, padding="post", truncating="post")
x_train.shape, y_train.shape

## <font color="#F48E16">Modelo</font>

In [ ]:
def build_function(embedding_units, rec_units, optimizer="Adam", activation="tanh", l1reg=0.0, additional_hidden_layers=0):
    input_layer = tf.keras.layers.Input(shape=(max_len,))
    embedding_layer = tf.keras.layers.Embedding(input_size, embedding_units, mask_zero=True)(input_layer)
    rec_layer = tf.keras.layers.LSTM(rec_units, activation=activation,
                                     kernel_regularizer=tf.keras.regularizers.l1(l1reg))(embedding_layer)
    output_layer = tf.keras.layers.Dense(2, activation="softmax")(rec_layer)

    model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

    return model

In [ ]:
embedding_units = 16
rec_units = 10
learning_rate = 1e-3
activation = "tanh"
l1reg = 0.0

optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
model = build_function(embedding_units, rec_units, optimizer, activation, l1reg)
model.summary()

In [ ]:
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=10, batch_size=128)

In [ ]:
model.save_weights(PATH + "models/IMDB/model__"+activation+"__l1reg_"+str(l1reg)+".weights.h5")

## <font color="#F48E16">XAI</font>

In [ ]:
N = 1000
x_train_small = x_train[:N]
y_train_small = y_train[:N]

In [ ]:
def gradient(model, X, y):
  embedding_model = tf.keras.Model(inputs=model.input, outputs=model.layers[1].output)
  X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
  E = embedding_model(X_tf)
  E = tf.Variable(E)

  with tf.GradientTape() as tape:
    tape.watch(E)
    rec_out = model.layers[2](E)
    y_pred = model.layers[3](rec_out)

    # Seleccionamos la salida correspondiente a la clase real
    indices = tf.stack([tf.range(tf.shape(y_pred)[0]), y], axis=1)
    y_class = tf.gather_nd(y_pred, indices)

  grads = tape.gradient(y_class, E)
  return grads.numpy()

In [ ]:
def gradientxinput(model, X, y):
  embedding_model = tf.keras.Model(inputs=model.input, outputs=model.layers[1].output)
  embedding = embedding_model(X)
  R = gradient(model, X, y) * embedding
  return np.sum(R.numpy(), axis=2)

In [ ]:
def epc(X, y, model, Rx, local=False, verbose=False):
  at = Rx.shape[-1]
  m = model.evaluate(X, y, batch_size=len(X), verbose=False)[-1]
  epc_all = []
  mmas_all = []
  mmenos_all = []
  iterador_at = tqdm(range(0, at, 2)) if verbose else range(0, at, 2)
  if not local: # Si es global
    for i in iterador_at:
        ar = at - i
        factor1 = (at - ar) / at
        umbral = np.sort(Rx)[i] # Usamos el mismo umbral para todo el dataset

        # Prueba 1
        X_prueba1 = X.copy()
        X_prueba1[:, Rx <= umbral] = 0
        mmas = model.evaluate(X_prueba1, y, batch_size=len(X_prueba1), verbose=False)[-1]

        # Prueba 2
        X_prueba2 = X.copy()
        X_prueba2[:, Rx >= umbral] = 0
        mmenos = model.evaluate(X_prueba2, y, batch_size=len(X_prueba2), verbose=False)[-1]

        factor2 = (mmas - mmenos) / m

        epc = factor1 * factor2
        epc_all.append(epc)
        mmas_all.append(mmas)
        mmenos_all.append(mmenos)

  else: # Si es local
    for i in iterador_at:
        ar = at - i
        factor1 = (at - ar) / at

        # Prueba 1
        X_prueba1 = X.copy()
        for j, (x, r) in enumerate(zip(X, Rx)): # Por cada instancia x hay una relevancia r
            umbral = np.sort(r)[i] # y tenemos que usar esa r para definir el umbral
            X_prueba1[j, r <= umbral] = 0
        mmas = model.evaluate(X_prueba1, y, batch_size=len(X_prueba1), verbose=False)[-1]

        # Prueba 2
        X_prueba2 = X.copy()
        for j, (x, r) in enumerate(zip(X, Rx)): # Lo mismo aquí con la prueba 2
            umbral = np.sort(r)[i]
            X_prueba2[j, r >= umbral] = 0
        mmenos = model.evaluate(X_prueba2, y, batch_size=len(X_prueba2), verbose=False)[-1]

        factor2 = (mmas - mmenos) / m

        epc = factor1 * factor2
        epc_all.append(epc)
        mmas_all.append(mmas)
        mmenos_all.append(mmenos)

  return np.array(epc_all), np.array(mmas_all), np.array(mmenos_all)

### <font color="#F48E16">EPC Global</font>

In [ ]:
time_ini = time()
R_gradientxinput = gradientxinput(model, x_train_small, y_train_small)
time_fin = time()
print(" > Tiempo de ejecución de Grad x Input:", time_fin - time_ini)
epc_gradientxinput, mmas_gradientxinput, mmenos_gradientxinput = epc(x_train_small, y_train_small, model, R_gradientxinput.reshape(-1, max_len), local=True, verbose=True)

In [ ]:
plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(mmas_gradientxinput, label="Prueba 1")
plt.plot(mmenos_gradientxinput, label="Prueba 2")
plt.legend()
plt.subplot(1,2,2)
plt.plot(epc_gradientxinput, label="EPC")
plt.legend()
plt.show()

### <font color="#F48E16">Evaluación local</font>

In [ ]:
# Cargar el diccionario de palabras
word_index = imdb.get_word_index()
index_to_word = {index + 3: word for word, index in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"

In [ ]:
# Función para decodificar una secuencia de IDs a palabras
def decode_review(sequence):
    return " ".join([index_to_word.get(i, "<UNK>") for i in sequence])

In [ ]:
i = 0 # 58
num_words = 10

R = R_gradientxinput[i]
R_sort = np.sort(R) # De menor a mayor
X = x_train[i].copy()
print(decode_review(X).replace("<PAD>", ""))
print(" > ", model.predict(X[None, :]))
print(" > ", y_train[i])
print()
X[R <= R_sort[199 - num_words]] = 0
print(decode_review(X))
print(decode_review(X).replace("<PAD>", ""))
print(" > ", model.predict(X[None, :]))
print(" > ", y_train[i])

### <font color="#F48E16">¿Cómo es el gradiente? ¿Hay vanishing gradient?</font>

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(R)
plt.show()

## <font color="#F48E16">Modelo con bloque Post-Recurrent (PRM)</font>

In [ ]:
def build_function_prm(embedding_units, rec_units, optimizer="Adam", activation="tanh", prm_activation="sigmoid", l1reg=0.0):
    input_layer = tf.keras.layers.Input(shape=(max_len,))
    embedding_layer = tf.keras.layers.Embedding(input_size, embedding_units, mask_zero=True)(input_layer)
    rec_layer = tf.keras.layers.LSTM(rec_units, activation=activation, return_sequences=True, # OJO: Return sequences a TRUE
                                     kernel_regularizer=tf.keras.regularizers.l1(l1reg))(embedding_layer)
    prm_layer = tf.keras.layers.Dense(1, activation=prm_activation, kernel_regularizer=tf.keras.regularizers.l1(l1reg),name='prm')(rec_layer)
    flatten_layer = tf.keras.layers.Flatten()(prm_layer)
    output_layer = tf.keras.layers.Dense(2, activation="softmax")(flatten_layer)

    model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

    return model

In [ ]:
def gradient_prm(model, X, y):
  embedding_model = tf.keras.Model(inputs=model.input, outputs=model.layers[1].output)
  X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
  E = embedding_model(X_tf)
  E = tf.Variable(E)

  with tf.GradientTape() as tape: # Como vamos capa a capa, tenemos que definir a mano los pasos en la cinta
    tape.watch(E)
    rec_out = model.layers[2](E)
    prm_out = model.layers[3](rec_out)
    flatten_out = model.layers[4](prm_out)
    y_pred = model.layers[5](flatten_out)

    # Seleccionamos la salida correspondiente a la clase real
    indices = tf.stack([tf.range(tf.shape(y_pred)[0]), y], axis=1)
    y_class = tf.gather_nd(y_pred, indices)

  grads = tape.gradient(y_class, E)
  return grads.numpy()

In [ ]:
def gradientxinput(model, X, y, use_prm=False):
  embedding_model = tf.keras.Model(inputs=model.input, outputs=model.layers[1].output)
  embedding = embedding_model(X)
  if use_prm:
    R = gradient_prm(model, X, y) * embedding
  else:
    R = gradient(model, X, y) * embedding
  return np.sum(R.numpy(), axis=2)

In [ ]:
embedding_units = 16
rec_units = 10
learning_rate = 1e-3
activation = "tanh"
prm_activation = "sigmoid"
l1reg = 0.0

optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
model = build_function_prm(embedding_units, rec_units, optimizer, activation, prm_activation, l1reg)
model.summary()

In [ ]:
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=10, batch_size=128)

In [ ]:
model.save_weights(PATH + "models/IMDB_PRM/model__"+activation+"__l1reg_"+str(l1reg)+".weights.h5")

### <font color="#F48E16">EPC Global</font>

In [ ]:
time_ini = time()
R_gradientxinput_prm = gradientxinput(model, x_train_small, y_train_small, use_prm=True)
time_fin = time()
print(" > Tiempo de ejecución de Grad x Input:", time_fin - time_ini)
epc_gradientxinput_prm, mmas_gradientxinput_prm, mmenos_gradientxinput_prm = epc(x_train_small, y_train_small, model, R_gradientxinput_prm.reshape(-1, max_len), local=True, verbose=True)

In [ ]:
plt.figure(figsize=(15,3))
plt.subplot(1,2,1)
plt.plot(mmas_gradientxinput_prm, label="Prueba 1")
plt.plot(mmenos_gradientxinput_prm, label="Prueba 2")
plt.legend()
plt.subplot(1,2,2)
plt.plot(epc_gradientxinput_prm, label="EPC")
plt.legend()
plt.show()

### <font color="#F48E16">Evaluación local</font>

In [ ]:
i = 0 # 58
num_words = 10

R = R_gradientxinput_prm[i]
R_sort = np.sort(R) # De menor a mayor
X = x_train[i].copy()
print(decode_review(X).replace("<PAD>", ""))
print(" > ", model.predict(X[None, :]))
print(" > ", y_train[i])
print()
X[R <= R_sort[199 - num_words]] = 0
print(decode_review(X))
print(decode_review(X).replace("<PAD>", ""))
print(" > ", model.predict(X[None, :]))
print(" > ", y_train[i])

### <font color="#F48E16">¿Cómo es el gradiente ahora?</font>

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(R)
plt.show()